## Segmentation & crack Detection

#### This notebook performs the actual identification and visualization of road cracks using the enhanced images generated in the previous step. It utilizes advanced segmentation and morphological operations to isolate defects.

In [6]:
import cv2
import numpy as np

# input Folder path
input_folder = "saved_enhanced/"
# output folder path
output_folder = "saved_detections/"
saved_count = 0 

print("Detection started. Results will be saved to 'saved_detections'")

while True:
    img_path = f"{input_folder}frame_{saved_count}.jpg"
    enhanced_img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    
    if enhanced_img is None:
        print(f"No more images. Total processed: {saved_count}")
        break
    
    display_img = cv2.cvtColor(enhanced_img, cv2.COLOR_GRAY2BGR)

    # Statistical Analysis
    # Calculating the brightness (Mean) and contrast (Standard Deviation) 
    # to understand how noisy or clear the image is.
    mean_val = np.mean(enhanced_img)
    std_val = np.std(enhanced_img)
    
    # Coefficient of Variation (Ratio):
    # This helps decide if we need heavy cleaning (Opening) or just connecting lines (Closing) based on the image's texture.
    ratio = std_val / mean_val if mean_val != 0 else 0

    # Binary Thresholding
    # Separating the crack pixels from the road. 
    # This creates a clear mask where the defects are isolated from the background.
    _, crack_mask = cv2.threshold(enhanced_img, 35, 255, cv2.THRESH_BINARY_INV)

    if ratio > 0.6: 
        # Extremely High Contrast (Clear Images)
        # The crack is already prominent, so we avoid 'Opening' to prevent losing detail.
        print("High Contrast. Skipping Opening phase.")
        
        # For visualization consistency, we map 'cleaned' to the original mask.
        cleaned = crack_mask.copy() 
        
        # Using a small 3x3 kernel for a gentle 'Closing' to bridge tiny gaps.
        kernel_cl = np.ones((3,3), np.uint8)
        connected = cv2.morphologyEx(crack_mask, cv2.MORPH_CLOSE, kernel_cl, iterations=1)
    
    elif 0.4 < ratio <= 0.6: 
        # Medium Contrast (Mild Noise/Texture)
        # Requires a balance between cleaning and detail preservation.
        print(" Medium Contrast. Applying light cleaning.")
        
        # 'Opening' with a tiny 2x2 kernel to remove micro-noise without erasing fine cracks.
        kernel_small = np.ones((2,2), np.uint8) 
        cleaned = cv2.morphologyEx(crack_mask, cv2.MORPH_OPEN, kernel_small, iterations=1)
        
        # 'Closing' with a 5x5 kernel to strengthen the crack structure.
        kernel_cl = np.ones((3,3), np.uint8)
        connected = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel_cl, iterations=2)
    
    else: 
        # Standard/Low Contrast (Fragmented or Noisy Images)
        # Cracks are likely broken into pixels; needs aggressive cleaning and merging.
        print("Standard/Low Contrast. Applying full cleaning.")
        
        # Standard 3x3 'Opening' to eliminate significant road surface artifacts.
        kernel = np.ones((3,3), np.uint8)
        cleaned = cv2.morphologyEx(crack_mask, cv2.MORPH_OPEN, kernel, iterations=2)
        
        # Strong 9x9 'Closing' to bridge large gaps and merge fragmented red-dots into lines.
        kernel_cl = np.ones((9,9), np.uint8)
        connected = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel_cl, iterations=2)
    

    # This function scans the binary mask and extracts the actual shapes of the cracks.
    # It turns raw pixel data into a list of mathematical objects (contours) 
    # that we can count, measure, and highlight on the final image.
    contours, _ = cv2.findContours(connected, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Creating a duplicate of the original image for visualization.
    # We draw our detections on this 'overlay'.
    # overlay = image.copy()

    # Detection Filter & Visualization
    # Iterating through each detected shape to filter out noise and highlight valid cracks.
    for cnt in contours:
        # Calculating the surface area of the detected shape
        area = cv2.contourArea(cnt)
        
        # Noise is not highlighted
        # This ignores tiny dots or textures that aren't road defects.
        if area > 150:
            # Drawing the detection:
            cv2.drawContours(display_img, [cnt], -1, (0, 0, 255), 2)
            
    # save detected frame
    save_path = f"{output_folder}detected_{saved_count}.jpg"
    cv2.imwrite(save_path, display_img) 

    # show output frame
    # cv2.imshow('Detection Result', display_img)
    
    saved_count += 1
    
    # ligic for exit loop
    if cv2.waitKey(0) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()
print("All detected frames saved successfully!")

Detection started. Results will be saved to 'saved_detections'
 Medium Contrast. Applying light cleaning.
 Medium Contrast. Applying light cleaning.
 Medium Contrast. Applying light cleaning.
 Medium Contrast. Applying light cleaning.
 Medium Contrast. Applying light cleaning.
 Medium Contrast. Applying light cleaning.
 Medium Contrast. Applying light cleaning.
 Medium Contrast. Applying light cleaning.
 Medium Contrast. Applying light cleaning.
 Medium Contrast. Applying light cleaning.
 Medium Contrast. Applying light cleaning.
 Medium Contrast. Applying light cleaning.
 Medium Contrast. Applying light cleaning.
 Medium Contrast. Applying light cleaning.
 Medium Contrast. Applying light cleaning.
High Contrast. Skipping Opening phase.
High Contrast. Skipping Opening phase.
No more images. Total processed: 17
All detected frames saved successfully!
